In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [14]:
base_model = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B" 

"""
alternative fine tuned version:
model_id = "abhi9ab/DeepSeek-R1-Distill-Llama-8B-finance-v1"
"""

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Make my own fine-tuned Lora-Adapter
# model = PeftModel.from_pretrained(model, "./your-trained-lora-adapter")

model = model.eval()
print("model loaded!")

Loading weights: 100%|██████████| 291/291 [00:20<00:00, 14.23it/s]


model loaded!


In [15]:
for headline in test_headlines:
    prompt = f"<｜begin of sentence｜><｜User｜>Analyze the market sentiment of this news: {headline}<｜Assistant｜><｜thought｜>"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.1,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    clean_output = full_output.replace('Ġ', ' ').replace('Ċ', '\n').replace('pre', '')
    
    print(f"\nFINANCIAL ANALYSIS FOR: {headline}")
    print("-" * 50)
    
    if "Result" in clean_output:
        parts = clean_output.split("Result")
        print(f"REASONING: {parts[0].strip()}")
        print(f"FINAL SENTIMENT: Result {parts[1].strip()}")
    else:
        print(clean_output)
    
    print("="*50)


FINANCIAL ANALYSIS FOR: NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026.
--------------------------------------------------
<beginofsentence><｜User｜>Analyzethemarketsentimentofthisnews:NVIDIAbeatsQ4earningsexpectationsbutwarnsofsupplychainconstraintsin2026.<｜Assistant｜><thought>Alright, so I'm trying to figure out how to analyze the market sentiment of this news: "NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 20-26." Okay, let's break this down step by step.

First, I know that NVIDIA is a big company in the tech industry, specifically in graphics processing units (GPUs) and AI. Their earnings reports can have a significant impact on their stock price and the broader market. So, the news that they've beaten their Q4 earnings expectations is a positive sign. Investors like when companies exceed expectations because it often means stronger than anticipated performance, which can lead to stock price increases.

But then

In [16]:
for headline in test_headlines:
    prompt = f"""
        You are a financial sentiment classifier.
        
        Analyze the headline and respond in EXACTLY this format:
        
        Sentiment: [Bullish, Bearish, or Neutral]
        Confidence: [number between 0 and 1]
        
        Rules:
        - Do NOT explain your reasoning
        - Output ONLY the two lines above
        - Be decisive
        
        Headline: {headline}
        """
            
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.1,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    clean_output = full_output.replace('Ġ', ' ').replace('Ċ', '\n').replace('pre', '')
    
    print(f"\nFINANCIAL ANALYSIS FOR: {headline}")
    print("-" * 50)
    
    if "Result" in clean_output:
        parts = clean_output.split("Result")
        print(f"REASONING: {parts[0].strip()}")
        print(f"FINAL SENTIMENT: Result {parts[1].strip()}")
    else:
        print(clean_output)
    
    print("="*50)


FINANCIAL ANALYSIS FOR: NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026.
--------------------------------------------------
Youareafinancialsentimentclassifier.AnalyzetheheadlineandrespondinEXACTLYthisformat:Sentiment:[Bullish,Bearish,orNeutral]Confidence:[numberbetween0and1]Rules:-DoNOTexplainyourreasoning-OutputONLYthetwolinesabove-BedecisiveHeadline:NVIDIAbeatsQ4earningsexpectationsbutwarnsofsupplychainconstraintsin2026. NVIDIA'sQ4earningsexceedexpectations,buttheyalsohighlightedsupplychainconstraintsfor2026. So, the headline is about NVIDIA beating earnings expectations but also warning about supply chain constraints in 2026. Sentiment: Bullish. Confidence: 0.85. NVIDIA's strong earnings and guidance suggest the company is performing well, which is positive for investors. However, the mention of supply chain issues in the future could be a concern. Overall, the sentiment is bullish as the company is doing well, but there's a note of caution rega